In [ ]:
import pandas as pd
import numpy as np
import flirt
import os
import jupyter
import ipywidgets

# 1. Load WESAD dataset

In [ ]:
# load data
df_acc = pd.read_parquet('data-input/dataset_wesad_wrist_acc.parquet')
df_bvp = pd.read_parquet('data-input/dataset_wesad_wrist_bvp.parquet')
df_eda = pd.read_parquet('data-input/dataset_wesad_wrist_eda.parquet')
df_temp = pd.read_parquet('data-input/dataset_wesad_wrist_temp.parquet')

# 2. Function for getting features from FLIRT

In [ ]:
def get_features_inner(df, columns_list, prefix, window_length, window_step_size, frequency):
     
    # we need to set a correct datetime index (nanoseconds calculated from frequency)
    # otherwise Flirt will create a wrong timeindex
    ns = '250000000N'
    if frequency == 64:
        ns = '15625000N'
    elif frequency == 32:
        ns = '31250000N'
    time_index = pd.date_range(start=0, periods=len(df), freq=ns)
    df = df.set_index(time_index)
    
    df = df[columns_list]
    df = df.dropna()

    features = flirt.get_acc_features(df,
                                      window_length=window_length, 
                                      window_step_size=window_step_size,
                                      data_frequency=frequency)
    features = features.add_prefix(prefix)
    return features

In [ ]:
def get_features(subject, label, df_acc, df_bvp, df_eda, df_temp, window_length, window_step_size):
    
    # calculate features
    acc_features = get_features_inner(df_acc, ['x', 'y', 'z'], 'acc_', window_length, window_step_size, 32)
    bvp_features = get_features_inner(df_bvp, ['BVP'], 'bvp_', window_length, window_step_size, 64)
    eda_features = get_features_inner(df_eda, ['EDA'], 'eda_', window_length, window_step_size, 4)
    temp_features = get_features_inner(df_temp, ['TEMP'], 'temp_', window_length, window_step_size, 4)

    # merge
    res = pd.concat([bvp_features, acc_features, eda_features, temp_features], axis=1)
    
    # add subject and label column
    res['subject'] = subject
    res['label'] = label

    return res

# 3. Calculate features for the whole dataset

In [ ]:
df_acc.shape

In [ ]:
df_acc

In [ ]:
iterlist = [(i, j)
    for i in df_acc.subject.unique()
    for j in df_acc.label.unique()]

In [ ]:
def get_all_chunks(df_acc, df_bvp, df_eda, df_temp, subject, label):
    df_acc_chunk = get_chunks(df_acc, subject, label)
    df_bvp_chunk = get_chunks(df_bvp, subject, label)
    df_eda_chunk = get_chunks(df_eda, subject, label)
    df_temp_chunk = get_chunks(df_temp, subject, label)
    return df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk

In [ ]:
def get_chunks(df, subject, label):
    df_chunk = df[df['subject'] == subject]
    df_chunk = df_chunk[df_chunk['label'] == label]
    df_chunk = df_chunk.drop(columns=['session', 'subject', 'label'])
    return df_chunk

In [ ]:
%%time

result_dfs = []

window_length = 60
window_step_size = 10

# loop over all subject-label combinations (all subjects have 2 sessions with 1 label each)
for (subject, label) in iterlist:
    
    df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk = get_all_chunks(df_acc, df_bvp, df_eda, df_temp, subject, label)
    
    res_df_chunks = get_features(subject, label, df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk, window_length, window_step_size)
    result_dfs.append(res_df_chunks)

res = pd.concat(result_dfs)

In [ ]:
res

In [ ]:

# handle NANs
res = res.dropna()

In [ ]:
res

In [ ]:
# store as parquet

if not os.path.isdir('data-input'):
    os.makedirs('data-input')

res.to_parquet('data-input/flirt-wesad-acc-bvp-eda-temp-'+str(window_length)+'-'+str(window_step_size)+'.parquet')